# Padel Highlights - GPU Inference Pipeline

Este notebook está diseñado para ejecutar la inferencia de jugadores (YOLO) y bola (TrackNet) utilizando la GPU de Google Colab. Esto reducirá el tiempo de procesamiento de horas a minutos.

### 1. Montar Google Drive
Recomendamos subir los vídeos a una carpeta en Google Drive para que los resultados se guarden permanentemente.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Clonar el Repositorio e Instalar Dependencias

In [2]:
!git clone https://github.com/rubenperezsoto/padel-highlights.git
%cd padel-highlights

# Instalar dependencias
!pip install ultralytics pandas torch torchvision opencv-python joblib scikit-learn torchsummary tensorboardX python-dotenv

Cloning into 'padel-highlights'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 54 (delta 9), reused 54 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 5.70 MiB | 4.18 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/padel-highlights
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.8 MB/s eta 0:00:00


### 3. Configurar TrackNetV2 y Pesos
Descargamos el repositorio de TrackNet y configuramos los pesos necesarios.

In [3]:
# Crear directorios
!mkdir -p external/weights

# Clonar TrackNetV2-pytorch
!git clone https://github.com/ChgygLin/TrackNetV2-pytorch external/TrackNetV2-pytorch

# Aplicar el parche para los pesos convertidos
%cd external/TrackNetV2-pytorch
!git apply tf2torch/diff.txt
%cd ../..

# Copiar pesos
!cp external/TrackNetV2-pytorch/tf2torch/track.pt external/weights/trackvnet.pt

Cloning into 'external/TrackNetV2-pytorch'...
remote: Enumerating objects: 487, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 487 (delta 40), reused 47 (delta 19), pack-reused 411 (from 1)
Receiving objects: 100% (487/487), 42.73 MiB | 10.73 MiB/s, done.
Resolving deltas: 100% (267/267), done.
/content/padel-highlights/external/TrackNetV2-pytorch
/content/padel-highlights


### 4. Configurar Variables de Entorno
Configuramos las rutas para que el código encuentre los modelos.

In [4]:
import os
os.environ['TRACKNETV2_ROOT'] = '/content/padel-highlights/external/TrackNetV2-pytorch'
os.environ['TRACKNETV2_WEIGHTS'] = '/content/padel-highlights/external/weights/trackvnet.pt'
os.environ['PYTHONPATH'] = '/content/padel-highlights'

### 5. Ejecutar Inferencia
Cambia la ruta de `--video` por la ubicación de tu vídeo en Google Drive.

In [5]:
# Ejemplo de ejecución
VIDEO_PATH = "/content/drive/MyDrive/padel/Padel Aurial arnau_capde vs elo_vice - Pau Capdevila Ribas (720p, h264).mp4" # <--- CAMBIA ESTO
OUTPUT_PATH = "/content/drive/MyDrive/padel/ticks_aurial.parquet"

!python3 -m src.padel.pipelines.run_inference_to_parquet \
    --video "{VIDEO_PATH}" \
    --output "{OUTPUT_PATH}"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Running inference (YOLO + TrackNet) on /content/drive/MyDrive/padel/Padel CNS bertro-capde vs elo-vice - Pau Capdevila Ribas (720p, h264).mp4...
TrackNet initialized successfully.
Processing YOLO detections...
Processing TrackNet detections...
Inference complete. Processed 102653 frames.
Resampling to 5.0 Hz and extracting features...
Success! Saved 17109 ticks to /content/drive/MyDrive/padel/ticks_resultado.parquet
